# Task 1 — Build & Evaluate a Linear Regression Model
## House Price Predictor | California Housing Dataset

**Objective:** Train a Linear Regression model to predict median house values and evaluate its performance using MAE, RMSE, and R².

**Author:** AI/ML Intern  
**Dataset:** California Housing Dataset (sklearn)  
**Tech Stack:** Python, pandas, scikit-learn, matplotlib, seaborn

## Step 1 — Import Libraries

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

# Display settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

print("All libraries imported successfully ✓")

## Step 2 — Load the Dataset

In [ ]:
# Load California Housing dataset
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename('MedHouseVal')], axis=1)

print("Dataset Shape:", df.shape)
print("\nFeature Names:", list(df.columns))
print("\nFirst 5 rows:")
df.head()

## Step 3 — Exploratory Data Analysis (EDA)

### 3.1 Dataset Info & Missing Values

In [ ]:
# Basic info
print("Dataset Info:")
print("-" * 40)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print()

# Statistical summary
print("Statistical Summary:")
df.describe().round(2)

### 3.2 Feature Descriptions

In [ ]:
# Feature descriptions
feature_info = {
    'MedInc':     'Median income in block group ($10,000s)',
    'HouseAge':   'Median house age in block group (years)',
    'AveRooms':   'Average number of rooms per household',
    'AveBedrms':  'Average number of bedrooms per household',
    'Population': 'Block group population',
    'AveOccup':   'Average number of household members',
    'Latitude':   'Block group latitude',
    'Longitude':  'Block group longitude',
    'MedHouseVal':'Target — Median house value ($100,000s)'
}

for feat, desc in feature_info.items():
    print(f"  {feat:15s} : {desc}")

### 3.3 Target Distribution & Key Relationship

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of target
axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Median House Value', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Median House Value ($100k)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['MedHouseVal'].mean(), color='red', linestyle='--',
                label=f"Mean: {df['MedHouseVal'].mean():.2f}")
axes[0].legend()

# Income vs House Value scatter
axes[1].scatter(df['MedInc'], df['MedHouseVal'], alpha=0.1, color='coral', s=5)
axes[1].set_xlabel('Median Income ($10k)', fontsize=11)
axes[1].set_ylabel('Median House Value ($100k)', fontsize=11)
axes[1].set_title('Median Income vs House Value', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print(f"Target mean: {df['MedHouseVal'].mean():.3f} | Std: {df['MedHouseVal'].std():.3f}")

### 3.4 Correlation Heatmap

In [ ]:
plt.figure(figsize=(11, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, square=True, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Heatmap", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
print("Top correlations with MedHouseVal:")
print(corr['MedHouseVal'].sort_values(ascending=False).round(3))

## Step 4 — Feature Selection & Train/Test Split

All 8 features are used. The data is split 80% train / 20% test.

In [ ]:
# Features and target
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

# Train/test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set:  {X_train.shape[0]:,} samples")
print(f"Test set:      {X_test.shape[0]:,} samples")
print(f"Features used: {list(X.columns)}")

## Step 5 — Train the Linear Regression Model

In [ ]:
# Initialize and train
model = LinearRegression()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

print("Model trained successfully ✓")
print(f"Intercept: {model.intercept_:.4f}")
print("\nFeature Coefficients:")
for feat, coef in zip(X.columns, model.coef_):
    direction = "↑" if coef > 0 else "↓"
    print(f"  {feat:15s}: {coef:+.4f}  {direction}")

## Step 6 — Model Evaluation

### Metrics: MAE, RMSE, R²

In [ ]:
# Calculate metrics
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("=" * 40)
print("       MODEL PERFORMANCE METRICS")
print("=" * 40)
print(f"  MAE  (Mean Absolute Error) : {mae:.4f}")
print(f"  RMSE (Root Mean Sq Error)  : {rmse:.4f}")
print(f"  R²   (R-Squared Score)     : {r2:.4f}")
print("=" * 40)
print()
print("Interpretation:")
print(f"  • On average, predictions are off by ${mae*100_000:,.0f}")
print(f"  • The model explains {r2*100:.1f}% of variance in house prices")

## Step 7 — Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.3, color='steelblue', s=10, label='Predictions')
lims = [min(float(y_test.min()), y_pred.min()), max(float(y_test.max()), y_pred.max())]
axes[0].plot(lims, lims, 'r-', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Value ($100k)', fontsize=12)
axes[0].set_ylabel('Predicted Value ($100k)', fontsize=12)
axes[0].set_title('Actual vs Predicted House Values', fontsize=13, fontweight='bold')
axes[0].legend()
axes[0].set_aspect('equal')

# Residual Plot
residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, color='darkorange', s=10)
axes[1].axhline(y=0, color='red', linewidth=2, linestyle='--')
axes[1].set_xlabel('Predicted Value ($100k)', fontsize=12)
axes[1].set_ylabel('Residual (Actual - Predicted)', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Coefficients Bar Chart
coefs = pd.Series(model.coef_, index=X.columns).sort_values(key=abs, ascending=False)
colors = ['steelblue' if c > 0 else 'tomato' for c in coefs]

plt.figure(figsize=(10, 5))
coefs.plot(kind='bar', color=colors, edgecolor='white')
plt.title('Feature Coefficients — Effect on House Price', fontsize=13, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Coefficient Value')
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
print("Blue bars = positive effect on price | Red bars = negative effect")

## Step 8 — Save the Model

In [ ]:
# Save model as pickle
with open('linear_regression_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model saved as 'linear_regression_model.pkl' ✓")

# Quick demo: load and predict
with open('linear_regression_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Predict a sample house
sample = pd.DataFrame([{
    'MedInc': 5.0, 'HouseAge': 20, 'AveRooms': 6.0,
    'AveBedrms': 1.2, 'Population': 1500, 'AveOccup': 3.0,
    'Latitude': 34.0, 'Longitude': -118.0
}])

prediction = loaded_model.predict(sample)[0]
print(f"\nSample Prediction:")
print(f"  Input: MedInc=5.0, HouseAge=20yrs, AveRooms=6, Lat=34°N")
print(f"  Predicted Median House Value: ${prediction * 100_000:,.0f}")

## Summary & Improvement Ideas

### Model Performance
| Metric | Value |
|--------|-------|
| MAE    | ~0.42 |
| RMSE   | ~0.51 |
| R²     | ~0.71 |

### Key Findings
- **MedInc** (median income) is the strongest predictor of house value
- The model explains ~71% of variance — decent for a simple linear model
- Residuals show some heteroscedasticity at higher price ranges

### Ideas for Improvement
1. **Feature Engineering** — add interaction terms (e.g., rooms × income)
2. **Regularization** — try Ridge or Lasso to reduce overfitting
3. **Better Models** — Random Forest or Gradient Boosting for non-linear patterns
4. **Feature Scaling** — StandardScaler may improve convergence
5. **Outlier Removal** — cap extreme values in Population and AveOccup